# Optimized Haar Cascade Algorithm for Accurate Face Detection

This notebook demonstrates the implementation and experimental evaluation of a Haar Cascade-based face detection system.

The objective is to investigate whether parameter fine-tuning can improve the performance of the default Haar Cascade face detector.

The parameters investigated in this experiment are:

- `scaleFactor`
- `minNeighbors`
- `minSize`
- `maxSize`

The system processes images in grayscale and evaluates detection performance using:

- Accuracy
- Precision
- Recall
- F1-Score

The experiment compares the default Haar Cascade configuration with a fine-tuned configuration.

## Research Objective

The objective of this experiment is to improve Haar Cascade face detection through systematic parameter fine-tuning.

The default Haar Cascade detector can produce false positives and missed detections depending on lighting conditions, image scale, background complexity, and face size.

This experiment investigates how modifying the detection parameters can improve detection reliability while retaining the lightweight computational characteristics of Haar Cascade.

In [ ]:
import cv2
import os
import json
import itertools
import numpy as np

from pathlib import Path
import matplotlib.pyplot as plt

print("OpenCV version:", cv2.__version__)

## Load Haar Cascade Classifier

The experiment uses OpenCV's pretrained frontal-face Haar Cascade classifier.

The classifier is stored as:

`haarcascade_frontalface_default.xml`

The classifier contains pretrained Haar-like features used for frontal face detection.

In [ ]:
CASCADE_PATH = "/content/models/haarcascade_frontalface_default.xml"

face_cascade = cv2.CascadeClassifier(CASCADE_PATH)

if face_cascade.empty():
    raise RuntimeError("Unable to load Haar Cascade classifier.")

print("Haar Cascade loaded successfully.")

## Dataset Organization

The experiment uses positive and negative images.

Positive images contain faces and are expected to produce at least one face detection.

Negative images do not contain the target face and should ideally produce no detections.

Expected directory structure:

```text
test_images/
├── positive/
│   ├── face1.jpg
│   ├── face2.jpg
│   └── ...
│
└── negative/
    ├── image1.jpg
    ├── image2.jpg
    └── ...

In [ ]:

## Cell 7 — Code

POSITIVE_DIR = "../test_images/positive"
NEGATIVE_DIR = "../test_images/negative"


def load_images(directory):
    images = []

    supported_extensions = (
        ".jpg",
        ".jpeg",
        ".png",
        ".bmp",
        ".tif",
        ".tiff"
    )

    if not os.path.exists(directory):
        print(f"Directory not found: {directory}")
        return images

    for filename in os.listdir(directory):

        if not filename.lower().endswith(supported_extensions):
            continue

        path = os.path.join(directory, filename)

        image = cv2.imread(path)

        if image is None:
            continue

        gray = cv2.cvtColor(
            image,
            cv2.COLOR_BGR2GRAY
        )

        images.append({
            "filename": filename,
            "image": gray
        })

    return images


positive_images = load_images(POSITIVE_DIR)
negative_images = load_images(NEGATIVE_DIR)

print("Positive images:", len(positive_images))
print("Negative images:", len(negative_images))

## Visualize a Sample Image

Before performing detection, a sample image is visualized to verify that the image loading and grayscale preprocessing stages are working correctly.

In [ ]:
if positive_images:

    sample = positive_images[0]["image"]

    plt.figure(figsize=(8, 6))
    plt.imshow(sample, cmap="gray")
    plt.title("Sample Grayscale Image")
    plt.axis("off")
    plt.show()

## Default Haar Cascade Detection

The first experiment uses the default Haar Cascade configuration.

This provides a baseline against which the fine-tuned configuration can be compared.

In [ ]:
DEFAULT_SCALE_FACTOR = 1.1
DEFAULT_MIN_NEIGHBORS = 5
DEFAULT_MIN_SIZE = (30, 30)
DEFAULT_MAX_SIZE = (500, 500)

In [ ]:
def detect_faces(
    image,
    scale_factor=1.1,
    min_neighbors=5,
    min_size=(30, 30),
    max_size=(500, 500)
):
    faces = face_cascade.detectMultiScale(
        image,
        scaleFactor=scale_factor,
        minNeighbors=min_neighbors,
        minSize=min_size,
        maxSize=max_size
    )

    return faces

In [ ]:
if positive_images:

    image = positive_images[0]["image"]

    faces = detect_faces(
        image,
        DEFAULT_SCALE_FACTOR,
        DEFAULT_MIN_NEIGHBORS,
        DEFAULT_MIN_SIZE,
        DEFAULT_MAX_SIZE
    )

    print("Faces detected:", len(faces))

## Visualize Default Detection

The detected face regions are displayed using bounding boxes.

This provides a qualitative view of the baseline detector's behavior.

In [ ]:
def draw_faces(image, faces):

    output = cv2.cvtColor(
        image,
        cv2.COLOR_GRAY2BGR
    )

    for (x, y, w, h) in faces:

        cv2.rectangle(
            output,
            (x, y),
            (x + w, y + h),
            (0, 255, 0),
            2
        )

        cv2.putText(
            output,
            "Face",
            (x, y - 10),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.7,
            (0, 255, 0),
            2
        )

    return output


if positive_images:

    result = draw_faces(
        positive_images[0]["image"],
        faces
    )

    plt.figure(figsize=(8, 6))
    plt.imshow(
        cv2.cvtColor(result, cv2.COLOR_BGR2RGB)
    )
    plt.title("Default Haar Cascade Detection")
    plt.axis("off")
    plt.show()

## Evaluation Method

For evaluation:

- A positive image is considered correctly detected when at least one face is detected.
- A positive image with no detection is considered a false negative.
- A negative image with one or more detections is considered a false positive.
- A negative image with no detections is considered a true negative.

These outcomes are used to calculate:

- Accuracy
- Precision
- Recall
- F1-Score

In [ ]:
def evaluate_parameters(
    positive_images,
    negative_images,
    scale_factor,
    min_neighbors,
    min_size,
    max_size
):

    tp = 0
    tn = 0
    fp = 0
    fn = 0

    # Positive samples
    for item in positive_images:

        faces = detect_faces(
            item["image"],
            scale_factor,
            min_neighbors,
            min_size,
            max_size
        )

        if len(faces) > 0:
            tp += 1
        else:
            fn += 1

    # Negative samples
    for item in negative_images:

        faces = detect_faces(
            item["image"],
            scale_factor,
            min_neighbors,
            min_size,
            max_size
        )

        if len(faces) > 0:
            fp += 1
        else:
            tn += 1

    total = tp + tn + fp + fn

    accuracy = (
        (tp + tn) / total
        if total > 0
        else 0
    )

    precision = (
        tp / (tp + fp)
        if (tp + fp) > 0
        else 0
    )

    recall = (
        tp / (tp + fn)
        if (tp + fn) > 0
        else 0
    )

    f1 = (
        2 * precision * recall /
        (precision + recall)
        if (precision + recall) > 0
        else 0
    )

    return {
        "accuracy": accuracy,
        "precision": precision,
        "recall": recall,
        "f1_score": f1,
        "tp": tp,
        "tn": tn,
        "fp": fp,
        "fn": fn
    }

## Haar Cascade Parameter Fine-Tuning

The main experiment searches for a parameter configuration that provides better detection performance.

The parameters investigated are:

### scaleFactor

Controls the image scale reduction between successive detection stages.

### minNeighbors

Controls how many neighboring detections are required before a candidate region is accepted.

### minSize

Defines the minimum size of a detected face.

### maxSize

Defines the maximum size of a detected face.

Different combinations are evaluated and the configuration producing the highest F1-score is selected.

In [ ]:
scale_factors = [
    1.05,
    1.10,
    1.15,
    1.20,
    1.25
]

min_neighbors_values = [
    3,
    4,
    5,
    6,
    7
]

min_sizes = [
    (20, 20),
    (30, 30),
    (40, 40)
]

max_sizes = [
    (300, 300),
    (500, 500),
    (1000, 1000)
]

print("Parameter search space created.")

In [ ]:
best_result = None
best_parameters = None

combinations = itertools.product(
    scale_factors,
    min_neighbors_values,
    min_sizes,
    max_sizes
)

for parameters in combinations:

    (
        scale_factor,
        min_neighbors,
        min_size,
        max_size
    ) = parameters

    result = evaluate_parameters(
        positive_images,
        negative_images,
        scale_factor,
        min_neighbors,
        min_size,
        max_size
    )

    if (
        best_result is None
        or result["f1_score"] > best_result["f1_score"]
    ):

        best_result = result

        best_parameters = {
            "scaleFactor": scale_factor,
            "minNeighbors": min_neighbors,
            "minSize": list(min_size),
            "maxSize": list(max_size)
        }


print("Best parameters:")
print(best_parameters)

print("\nBest metrics:")
print(best_result)

In [ ]:
os.makedirs("../models", exist_ok=True)

best_configuration = {
    "model": "OpenCV Haar Cascade",
    "classifier": "models/haarcascade_frontalface_default.xml",
    "parameters": best_parameters,
    "metrics": best_result
}

with open(
    "../models/best_params.json",
    "w"
) as file:

    json.dump(
        best_configuration,
        file,
        indent=4
    )

print("Best parameters saved.")

In [ ]:
## Default vs Fine-Tuned Configuration

The final experiment compares the baseline configuration against the best configuration identified during parameter search.

The research reports the following performance for the Haar Cascade experiment:

| Model | Accuracy | Precision | Recall | F1-Score |
|---|---:|---:|---:|---:|
| Default Haar Cascade | 22% | 0.21 | 0.20 | 0.21 |
| Fine-Tuned Haar Cascade | 90% | 0.89 | 0.92 | 0.90 |

These values represent the reported experimental results from the research work.

The values generated by this notebook should be treated as the results of the dataset actually supplied to the notebook.

In [ ]:
print("=" * 50)
print("FINE-TUNED HAAR CASCADE RESULTS")
print("=" * 50)

print(
    f"Accuracy : "
    f"{best_result['accuracy'] * 100:.2f}%"
)

print(
    f"Precision: "
    f"{best_result['precision']:.4f}"
)

print(
    f"Recall   : "
    f"{best_result['recall']:.4f}"
)

print(
    f"F1-Score : "
    f"{best_result['f1_score']:.4f}"
)

print("\nConfusion Matrix")

print("True Positive :", best_result["tp"])
print("True Negative :", best_result["tn"])
print("False Positive:", best_result["fp"])
print("False Negative:", best_result["fn"])

## Conclusion

The experiment demonstrates how parameter fine-tuning can improve the performance of a Haar Cascade face detector.

The main parameters investigated were:

- `scaleFactor`
- `minNeighbors`
- `minSize`
- `maxSize`

The fine-tuning process searches multiple parameter combinations and selects the configuration that produces the strongest evaluation performance.

The research reports an improvement from 22% accuracy for the default Haar Cascade configuration to 90% for the fine-tuned configuration, together with improvements in precision, recall, and F1-score.

The optimized approach retains the lightweight nature of Haar Cascade while improving detection reliability.

Potential future improvements include combining Haar Cascade with CNN/YOLO-based methods, adaptive parameter selection, increasing dataset diversity, and GPU/parallel processing.